# Supply chain reasoning: SQL, Python and RelationalAI together

This notebook asks a series of increasingly hard supply-chain questions about one small dataset, and answers each one three ways:

- **SQL**, the right tool for storing, filtering, joining and aggregating;
- **Python** (pandas, NetworkX), flexible, but each question becomes custom code that runs in the notebook's memory;
- **RelationalAI (PyRel)**, a semantic model of the business that is defined once and reused for rules, recursion, graph algorithms and optimisation.

The point is not that one replaces the others. Each chapter says plainly where SQL is enough, what the Python alternative costs, and what the model adds. All three run on the same Snowflake tables.

| Chapter | Question | Planted case |
|---|---|---|
| 0 | What's in the data? | |
| 1 | Model the domain (ontology) | |
| 2 | Which products contain the control board, at any depth? | 1 |
| 3 | A tier-3 supplier fails: what is affected? | 2, 4 |
| 4 | Which products can we still build from stock? | 3 |
| 5 | Which suppliers are structurally critical? | |
| 6 | Which orders should get the scarce sensors? | 2 |
| 7 | Report the results with SQL | |

## Before you start

- Run `scripts/load_data.sql` and `scripts/admin_setup.sql`. These create `RAI_QUICKSTART.SUPPLY_CHAIN`, the role `RAI_QS_ROLE` and the external access integration `RAI_NOTEBOOK_ACCESS`.
- Run this notebook on **Container Runtime** with `RAI_NOTEBOOK_ACCESS` enabled, as `RAI_QS_ROLE`.
- Notebook `01_relationalai_setup.ipynb` explains the setup step by step. If you have run it, the install below is already done.

**Install, then restart the kernel.** Installing `relationalai` upgrades `protobuf`, and the kernel keeps the old version loaded until it restarts. Run the next cell once, restart the kernel, then continue from the cell after it.

In [ ]:
%pip install -q "relationalai>=1.30.1,<2.0" networkx
# Now restart the kernel, then continue with the next cell.

In [ ]:
import importlib.metadata as md

import networkx as nx
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql("USE SCHEMA RAI_QUICKSTART.SUPPLY_CHAIN").collect()
print("relationalai", md.version("relationalai"), "| protobuf", md.version("protobuf"))


def sql(query):
    """Run SQL and return a pandas DataFrame with lower-case column names."""
    df = session.sql(query).to_pandas()
    df.columns = [c.lower() for c in df.columns]
    return df


def table(name):
    """Pull a whole table into pandas: what the Python alternative does."""
    return sql(f"SELECT * FROM {name}")


# The planted answers (also in data/expected_results.json). Each chapter checks against them.
EXPECTED = {
    "control_board_products": ["PR-01", "PR-02", "PR-03", "PR-04", "PR-05", "PR-06", "PR-07", "PR-08"],
    "failed_suppliers": ["S-T1-02", "S-T1-05", "S-T2-03", "S-T2-08", "S-T3-07"],
    "parts_without_supplier": ["P-141"],
    "affected_orders": 568,
    "affected_order_value": 6840136.49,
    "buildable_without_substitutes": ["PR-02", "PR-05", "PR-10", "PR-13", "PR-15", "PR-16", "PR-17", "PR-18", "PR-19"],
    "buildable_with_substitutes": ["PR-02", "PR-04", "PR-05", "PR-09", "PR-10", "PR-13", "PR-15", "PR-16", "PR-17", "PR-18", "PR-19"],
}

## 0. What's in the data

A fictional manufacturer builds **20 IoT devices** (trackers, meters, controllers). Each device is made of modules, which are made of components: **315 parts**, in a bill of materials (BOM) 3 to 5 levels deep. Modules are shared between products. **30 tier-1 suppliers** sell the components; they in turn depend on **50 suppliers in tiers 2–4**. There are **1,500 open orders**.

```
Customer ─▶ Order ─▶ Product ─▶ finished part ─▶ modules ─▶ … ─▶ components ◀─ tier-1 supplier ─▶ tier 2 ─▶ tier 3 ─▶ tier 4
```

Two networks meet at the component: the product structure runs *down*, the supply network runs *up*. The interesting questions cross both.

Most of the data is random. **Four cases are planted** so we know what to look for:

| Case | What is planted | What we look for |
|---|---|---|
| 1 | Control board `P-040` sits inside 8 products, in 2 of them one level deeper | All products that contain it, at any depth |
| 2 | Sensor `P-141` looks dual-sourced (`S-T1-02`, `S-T1-05`), but both suppliers depend on tier-3 supplier `S-T3-07` | What `S-T3-07` failing really hits |
| 3 | Product `PR-04` lacks only radio chip `P-121`; substitute `P-122` is in stock | Which products are buildable once substitutes count |
| 4 | Suppliers `S-T2-11` and `S-T2-14` depend on each other | Traversals that still finish despite the loop |

`data/README.md` describes each case in full.

In [ ]:
# SQL is the right tool to look around.
sql("""
SELECT 'PARTS' AS table_name, COUNT(*) AS row_count FROM PARTS
UNION ALL SELECT 'PRODUCTS', COUNT(*) FROM PRODUCTS
UNION ALL SELECT 'BOM', COUNT(*) FROM BOM
UNION ALL SELECT 'SUBSTITUTES', COUNT(*) FROM SUBSTITUTES
UNION ALL SELECT 'INVENTORY', COUNT(*) FROM INVENTORY
UNION ALL SELECT 'SUPPLIERS', COUNT(*) FROM SUPPLIERS
UNION ALL SELECT 'SUPPLIER_PARTS', COUNT(*) FROM SUPPLIER_PARTS
UNION ALL SELECT 'SUPPLIER_DEPENDENCIES', COUNT(*) FROM SUPPLIER_DEPENDENCIES
UNION ALL SELECT 'CUSTOMERS', COUNT(*) FROM CUSTOMERS
UNION ALL SELECT 'ORDERS', COUNT(*) FROM ORDERS
""")

In [ ]:
# One product's bill of materials: the Fleet telematics unit (PR-04), drawn as an indented tree.
# The BOM is a DAG (no cycles), so this recursion terminates; the depth limit is a safety net.
tree = sql("""
WITH RECURSIVE tree (part_id, qty, depth, path) AS (
    SELECT b.child_part_id, b.qty_per, 1, b.child_part_id
    FROM BOM b JOIN PRODUCTS p ON p.part_id = b.parent_part_id WHERE p.product_id = 'PR-04'
    UNION ALL
    SELECT b.child_part_id, b.qty_per, t.depth + 1, t.path || '>' || b.child_part_id
    FROM tree t JOIN BOM b ON b.parent_part_id = t.part_id
    WHERE t.depth < 10
)
SELECT t.depth, t.qty, t.part_id, p.part_name, p.part_type, i.on_hand_qty, t.path
FROM tree t JOIN PARTS p USING (part_id) JOIN INVENTORY i USING (part_id)
ORDER BY t.path
""")
for r in tree.itertuples():
    print(f"{'    ' * (r.depth - 1)}{r.qty} x {r.part_id} {r.part_name:<28} [{r.part_type.lower()}, stock {r.on_hand_qty}]")

## 1. Model the domain (ontology)

**This step is required.** RelationalAI doesn't reason over tables directly. It reasons over a *semantic model*, also called an ontology. The model has three parts:
- the business **concepts**: part, supplier, order, …
- the **relationships** between them: a part contains parts, a supplier depends on a supplier, …
- a **mapping** from each Snowflake table onto those concepts.

Every later chapter queries and extends this one model.

**The trade-off, stated fairly:** SQL and pandas need no model up front, so this is extra work. It pays off because chapters 2–6 reuse the same concepts and rules instead of restating join paths each time. For a single fixed-shape question, SQL without a model is simpler.

**How it's built:**
1. **Concepts with an identity** (`identify_by`): anything with its own key.
2. **Properties** for single-valued facts: a part's cost, an order's customer.
3. **Relationships** for links. Same-type links get role names: `{Part:parent} contains {Part:child}`.
4. **A junction concept, `Supply`**, for a supplier–part link that carries several values (capacity, lead time, price).
5. **Mapping**: `model.Table(...)`, then `Concept.new(...)` from the table that owns the key and `Concept.lookup(...)` to resolve foreign keys.

Types follow the column types: `NUMBER(p,0)` → `Integer`, `NUMBER(p,2)` → `Float`, `DATE` → `Date`.

The same code is in `notebooks/supply_chain_ontology.py`, if you'd rather import it.

In [ ]:
from relationalai.semantics import Date, Float, Integer, Model, String
from relationalai.semantics.std import aggregates as aggs

SCHEMA = "RAI_QUICKSTART.SUPPLY_CHAIN"
model = Model("SupplyChain")  # picks up the notebook's session: no config file, no credentials

# --- Concepts: things with their own identity -------------------------------------------
Part = model.Concept("Part", identify_by={"id": String})
Product = model.Concept("Product", identify_by={"id": String})
Supplier = model.Concept("Supplier", identify_by={"id": String})
Customer = model.Concept("Customer", identify_by={"id": String})
Order = model.Concept("Order", identify_by={"id": String})
Supply = model.Concept("Supply", identify_by={"supplier": Supplier, "part": Part})  # supplier-part pair

# --- Properties: one value per entity ---------------------------------------------------
Part.name = model.Property(f"{Part} is named {String:name}")
Part.kind = model.Property(f"{Part} is of kind {String:kind}")  # FINISHED | ASSEMBLY | COMPONENT
Part.unit_cost = model.Property(f"{Part} costs {Float:unit_cost}")
Part.on_hand = model.Property(f"{Part} has {Integer:on_hand} in stock")

Product.name = model.Property(f"{Product} is named {String:name}")
Product.list_price = model.Property(f"{Product} sells for {Float:list_price}")
Product.top_part = model.Property(f"{Product} is built as {Part:top_part}")

Supplier.name = model.Property(f"{Supplier} is named {String:name}")
Supplier.tier = model.Property(f"{Supplier} is in tier {Integer:tier}")
Supplier.country = model.Property(f"{Supplier} is located in {String:country}")
Supplier.region = model.Property(f"{Supplier} is in region {String:region}")

Supply.capacity_per_week = model.Property(f"{Supply} can deliver {Integer:capacity_per_week} per week")
Supply.lead_time_days = model.Property(f"{Supply} has a lead time of {Integer:lead_time_days} days")
Supply.unit_price = model.Property(f"{Supply} is priced at {Float:unit_price}")

Customer.name = model.Property(f"{Customer} is named {String:name}")
Customer.region = model.Property(f"{Customer} is in region {String:region}")

Order.customer = model.Property(f"{Order} is placed by {Customer:customer}")
Order.product = model.Property(f"{Order} is for {Product:product}")
Order.quantity = model.Property(f"{Order} asks for {Integer:quantity} units")
Order.due_date = model.Property(f"{Order} is due on {Date:due_date}")
Order.value = model.Property(f"{Order} is worth {Float:value}")

# --- Links between entities (same-type slots get role names) ---------------------------
Part.contains = model.Property(f"{Part:parent} contains {Part:child} {Integer:qty} times")
Part.substitute = model.Property(f"{Part:original} can be replaced by {Part:substitute} at ratio {Float:ratio}")
Supplier.depends_on = model.Relationship(f"{Supplier:buyer} depends on {Supplier:upstream}")

# --- Mapping: which table populates what -----------------------------------------------
parts = model.Table(f"{SCHEMA}.PARTS")
products = model.Table(f"{SCHEMA}.PRODUCTS")
bom = model.Table(f"{SCHEMA}.BOM")
substitutes = model.Table(f"{SCHEMA}.SUBSTITUTES")
inventory = model.Table(f"{SCHEMA}.INVENTORY")
suppliers = model.Table(f"{SCHEMA}.SUPPLIERS")
supplier_parts = model.Table(f"{SCHEMA}.SUPPLIER_PARTS")
supplier_deps = model.Table(f"{SCHEMA}.SUPPLIER_DEPENDENCIES")
customers = model.Table(f"{SCHEMA}.CUSTOMERS")
orders = model.Table(f"{SCHEMA}.ORDERS")

model.define(p := Part.new(id=parts.PART_ID),
             p.name(parts.PART_NAME), p.kind(parts.PART_TYPE), p.unit_cost(parts.UNIT_COST))
model.define(Part.lookup(id=inventory.PART_ID).on_hand(inventory.ON_HAND_QTY))
model.define(pr := Product.new(id=products.PRODUCT_ID),
             pr.name(products.PRODUCT_NAME), pr.list_price(products.LIST_PRICE),
             pr.top_part(Part.lookup(id=products.PART_ID)))
model.define(Part.lookup(id=bom.PARENT_PART_ID).contains(Part.lookup(id=bom.CHILD_PART_ID), bom.QTY_PER))
model.define(Part.lookup(id=substitutes.PART_ID).substitute(
    Part.lookup(id=substitutes.SUBSTITUTE_PART_ID), substitutes.RATIO))
model.define(s := Supplier.new(id=suppliers.SUPPLIER_ID),
             s.name(suppliers.SUPPLIER_NAME), s.tier(suppliers.TIER),
             s.country(suppliers.COUNTRY), s.region(suppliers.REGION))
model.define(Supplier.lookup(id=supplier_deps.SUPPLIER_ID).depends_on(
    Supplier.lookup(id=supplier_deps.DEPENDS_ON_SUPPLIER_ID)))
model.define(sp := Supply.new(supplier=Supplier.lookup(id=supplier_parts.SUPPLIER_ID),
                              part=Part.lookup(id=supplier_parts.PART_ID)),
             sp.capacity_per_week(supplier_parts.CAPACITY_PER_WEEK),
             sp.lead_time_days(supplier_parts.LEAD_TIME_DAYS),
             sp.unit_price(supplier_parts.UNIT_PRICE))
model.define(c := Customer.new(id=customers.CUSTOMER_ID),
             c.name(customers.CUSTOMER_NAME), c.region(customers.REGION))
model.define(o := Order.new(id=orders.ORDER_ID),
             o.customer(Customer.lookup(id=orders.CUSTOMER_ID)),
             o.product(Product.lookup(id=orders.PRODUCT_ID)),
             o.quantity(orders.QUANTITY), o.due_date(orders.DUE_DATE), o.value(orders.ORDER_VALUE))

In [ ]:
# Validate the model: every concept should hold as many entities as its table has rows.
# The FIRST query takes several minutes: RelationalAI starts an engine and indexes the 10 tables.
# After that, queries take seconds.
expected_counts = {"Part": 315, "Product": 20, "Supplier": 80, "Supply": 483, "Customer": 100, "Order": 1500}
for name, concept in [("Part", Part), ("Product", Product), ("Supplier", Supplier),
                      ("Supply", Supply), ("Customer", Customer), ("Order", Order)]:
    n = int(model.select(aggs.count(concept).alias("n")).to_df()["n"].iloc[0])
    print(f"{name:<9} {n:>5}   expected {expected_counts[name]}")
    assert n == expected_counts[name]

from relationalai.semantics import inspect
print(inspect.schema(model))

## 2. Which products contain the control board, at any depth? (case 1)

A one-level join finds the six products that use `P-040` directly, and misses the two that use it inside another module. You can't know in advance how deep to look.

**SQL** handles this with a recursive common table expression (CTE). That works well here, because the BOM has no cycles.

In [ ]:
direct = sql("""
SELECT p.product_id FROM PRODUCTS p JOIN BOM b ON b.parent_part_id = p.part_id
WHERE b.child_part_id = 'P-040' ORDER BY 1""")
print("one level only:", list(direct.product_id))

any_depth = sql("""
WITH RECURSIVE explode (product_id, part_id) AS (
    SELECT p.product_id, b.child_part_id FROM PRODUCTS p JOIN BOM b ON b.parent_part_id = p.part_id
    UNION ALL
    SELECT e.product_id, b.child_part_id FROM explode e JOIN BOM b ON b.parent_part_id = e.part_id
)
SELECT DISTINCT product_id FROM explode WHERE part_id = 'P-040' ORDER BY 1""")
print("any depth:     ", list(any_depth.product_id))
assert list(any_depth.product_id) == EXPECTED["control_board_products"]

In [ ]:
# Python alternative: pull the BOM into memory and walk it with NetworkX.
bom_df, products_df = table("BOM"), table("PRODUCTS")
bom_graph = nx.DiGraph(list(zip(bom_df.parent_part_id, bom_df.child_part_id)))
py_result = sorted(r.product_id for r in products_df.itertuples()
                   if "P-040" in nx.descendants(bom_graph, r.part_id))
print(py_result)
assert py_result == EXPECTED["control_board_products"]

**PyRel:** define "includes at some level" once, as two rules: a base case and a recursive case. After that, *any* question about containment is a short query. Unlike the CTE, the rule isn't tied to this one question. Chapters 3, 4 and 6 reuse it.

In [ ]:
Part.includes = model.Relationship(f"{Part:assembly} includes {Part:part} at some level")
a, b, c, qty = Part.ref(), Part.ref(), Part.ref(), Integer.ref()
model.where(a.contains(b, qty)).define(a.includes(b))                         # base case
model.where(a.includes(b), b.contains(c, qty)).define(a.includes(c))          # recursive case

top, board = Part.ref(), Part.ref()
rai_result = model.where(Product.top_part(top), top.includes(board), board.id == "P-040").select(
    Product.id.alias("product"), Product.name.alias("name")).to_df().sort_values("product")
display(rai_result)
assert list(rai_result["product"]) == EXPECTED["control_board_products"]

**Verdict:** all three approaches get the right answer. SQL is perfectly fine for this question on its own. What the model adds is a reusable definition of "contains, at any depth" that later chapters build on.

## 3. A tier-3 supplier fails: what is affected? (cases 2 and 4)

Supplier `S-T3-07` goes down. Anyone who relies on it, directly or through other tiers, goes down too. Then we check which parts are left with no working supplier, which products contain those parts, and which orders are hit. That is two traversals of unknown depth, chained together: *up* the supplier tiers, then *down* the BOM.

**The supplier network has a cycle** (`S-T2-11 ↔ S-T2-14`). A naive recursive CTE never finishes on it:

```sql
-- Do NOT run: loops forever on the planted cycle.
WITH RECURSIVE upstream (supplier_id, depends_on_id) AS (
    SELECT supplier_id, depends_on_supplier_id FROM SUPPLIER_DEPENDENCIES WHERE supplier_id = 'S-T1-11'
    UNION ALL
    SELECT u.supplier_id, d.depends_on_supplier_id
    FROM upstream u JOIN SUPPLIER_DEPENDENCIES d ON d.supplier_id = u.depends_on_id
)
SELECT DISTINCT depends_on_id FROM upstream;
```

When we tested it while building this guide, it was still running after an hour. SQL needs a hand-written guard: track the visited path and cap the depth.

In [ ]:
# SQL, with the guard: carry the visited path, skip suppliers already on it, cap the depth.
impact_sql = sql("""
WITH RECURSIVE down_suppliers (supplier_id, path, depth) AS (
    SELECT supplier_id, ARRAY_CONSTRUCT(supplier_id), 0 FROM SUPPLIERS WHERE supplier_id = 'S-T3-07'
    UNION ALL
    SELECT d.supplier_id, ARRAY_APPEND(u.path, d.supplier_id), u.depth + 1
    FROM down_suppliers u JOIN SUPPLIER_DEPENDENCIES d ON d.depends_on_supplier_id = u.supplier_id
    WHERE NOT ARRAY_CONTAINS(d.supplier_id::VARIANT, u.path) AND u.depth < 10
),
down AS (SELECT DISTINCT supplier_id FROM down_suppliers),
unsourced AS (                       -- parts whose every supplier is down
    SELECT part_id FROM SUPPLIER_PARTS
    GROUP BY part_id HAVING COUNT_IF(supplier_id NOT IN (SELECT supplier_id FROM down)) = 0
),
explode (product_id, part_id) AS (   -- BOM again, as in chapter 2
    SELECT p.product_id, b.child_part_id FROM PRODUCTS p JOIN BOM b ON b.parent_part_id = p.part_id
    UNION ALL
    SELECT e.product_id, b.child_part_id FROM explode e JOIN BOM b ON b.parent_part_id = e.part_id
),
hit AS (SELECT DISTINCT product_id FROM explode WHERE part_id IN (SELECT part_id FROM unsourced))
SELECT (SELECT ARRAY_AGG(supplier_id) WITHIN GROUP (ORDER BY supplier_id) FROM down) AS down_suppliers,
       (SELECT ARRAY_AGG(part_id) FROM unsourced) AS parts_without_supplier,
       COUNT(*) AS affected_orders, SUM(order_value) AS affected_value
FROM ORDERS WHERE product_id IN (SELECT product_id FROM hit)
""")
impact_sql.T

In [ ]:
# Python alternative: NetworkX handles the cycle for us, but we glue two graphs together by hand.
deps_df, sp_df, orders_df = table("SUPPLIER_DEPENDENCIES"), table("SUPPLIER_PARTS"), table("ORDERS")
supplier_graph = nx.DiGraph(list(zip(deps_df.supplier_id, deps_df.depends_on_supplier_id)))

down = {"S-T3-07"} | nx.ancestors(supplier_graph, "S-T3-07")          # everyone who relies on it
working = sp_df[~sp_df.supplier_id.isin(down)]
unsourced = set(sp_df.part_id) - set(working.part_id)
hit = [r.product_id for r in products_df.itertuples() if nx.descendants(bom_graph, r.part_id) & unsourced]
affected = orders_df[orders_df.product_id.isin(hit)]
print(sorted(down), sorted(unsourced), len(affected), round(affected.order_value.sum(), 2))
assert sorted(down) == EXPECTED["failed_suppliers"]
assert len(affected) == EXPECTED["affected_orders"]

In [ ]:
# PyRel: the same recursive pattern as chapter 2, now on suppliers. No cycle guard needed:
# rules derive *facts*, and a fact that is already known is not derived again, so the loop ends.
Supplier.relies_on = model.Relationship(f"{Supplier:buyer} relies on {Supplier:upstream} at some tier")
s1, s2, s3 = Supplier.ref(), Supplier.ref(), Supplier.ref()
model.where(s1.depends_on(s2)).define(s1.relies_on(s2))
model.where(s1.relies_on(s2), s2.depends_on(s3)).define(s1.relies_on(s3))

FAILED = "S-T3-07"
Supplier.is_down = model.Relationship(f"{Supplier} is down")
model.where(Supplier.id == FAILED).define(Supplier.is_down())
failed = Supplier.ref()
model.where(Supplier.relies_on(failed), failed.id == FAILED).define(Supplier.is_down())

# A part has no working supplier if it has suppliers, and none of them is still up.
Part.has_no_supplier = model.Relationship(f"{Part} has no working supplier")
any_supply, other = Supply.ref(), Supply.ref()
model.where(any_supply.part == Part,
            model.not_(other.part == Part, model.not_(other.supplier.is_down()))).define(Part.has_no_supplier())

# Reuse Part.includes from chapter 2 to go from parts to products to orders.
top, part = Part.ref(), Part.ref()
affected_rai = model.where(Order.product(Product), Product.top_part(top), top.includes(part),
                           part.has_no_supplier()).select(
    Order.id.alias("order"), Product.id.alias("product"), Order.value.alias("value")).to_df().drop_duplicates("order")

down_rai = sorted(model.where(Supplier.is_down()).select(Supplier.id.alias("s")).to_df()["s"])
print("down:", down_rai)
print("orders hit:", len(affected_rai), "value:", round(affected_rai["value"].sum(), 2))
display(affected_rai.groupby("product").agg(orders=("order", "count"), value=("value", "sum")))
assert down_rai == EXPECTED["failed_suppliers"]
assert len(affected_rai) == EXPECTED["affected_orders"]
assert round(affected_rai["value"].sum(), 2) == EXPECTED["affected_order_value"]

# The cycle (case 4): S-T1-11 reaches both S-T2-11 and S-T2-14, and the query simply finishes.
up = Supplier.ref()
print(sorted(model.where(Supplier.id == "S-T1-11", Supplier.relies_on(up)).select(up.id.alias("u")).to_df()["u"]))

**Verdict:** a tier-1 view shows two suppliers for sensor `P-141`. Only by following both networks do we see a single tier-3 dependency behind them, which puts 568 orders worth 6.84M at risk.

| | SQL | Python | PyRel |
|---|---|---|---|
| Cycle | Hand-written path tracking + depth cap; forget it and the query runs forever | Handled by NetworkX | Handled by the rule semantics |
| Effort | One long query: two recursions and set logic | Two in-memory graphs plus glue code | Three short rules, reusing `includes` from chapter 2 |
| Reuse | Copy the CTE into the next query | Copy the functions | `is_down`, `has_no_supplier` are now part of the model |

## 4. Which products can we still build from stock? (case 3)

A part is available if it's in stock, or it's a module whose parts are all available, or it's a component with an approved substitute in stock. The rule applies at every level of the BOM.

**An honest note on the formulation.** The natural phrasing, "*all* children are available", is a negation inside a recursion ("no child is unavailable"). PyRel doesn't allow recursion through negation, and most SQL engines restrict negation inside recursive CTEs as well. Both need the same trick: turn the rule around into a positive one. A part is **blocked** if it's out of stock and *any* of its children is blocked. A product is buildable if its top part is not blocked; that final negation sits outside the recursion. Once reformulated this way, SQL can do it in one query too. The real difference is reuse: in chapter 6 the blocked-or-not facts feed straight into an optimisation.

In [ ]:
# SQL, positive formulation: propagate "blocked" upwards, then take the complement.
buildable_sql = sql("""
WITH RECURSIVE short AS (            -- out-of-stock components without a substitute in stock
    SELECT p.part_id FROM PARTS p JOIN INVENTORY i USING (part_id)
    WHERE p.part_type = 'COMPONENT' AND i.on_hand_qty = 0
      AND NOT EXISTS (SELECT 1 FROM SUBSTITUTES s JOIN INVENTORY si ON si.part_id = s.substitute_part_id
                      WHERE s.part_id = p.part_id AND si.on_hand_qty > 0)
),
blocked (part_id) AS (
    SELECT part_id FROM short
    UNION ALL
    SELECT b.parent_part_id FROM blocked x JOIN BOM b ON b.child_part_id = x.part_id
    JOIN INVENTORY i ON i.part_id = b.parent_part_id WHERE i.on_hand_qty = 0
)
SELECT product_id FROM PRODUCTS WHERE part_id NOT IN (SELECT part_id FROM blocked) ORDER BY 1
""")
print(list(buildable_sql.product_id))
assert list(buildable_sql.product_id) == EXPECTED["buildable_with_substitutes"]

In [ ]:
# Python alternative: a memoised recursive function (Python has no restriction on negation).
parts_df, inv_df, subs_df = table("PARTS"), table("INVENTORY"), table("SUBSTITUTES")
stock = dict(zip(inv_df.part_id, inv_df.on_hand_qty))
kind = dict(zip(parts_df.part_id, parts_df.part_type))
alternatives = subs_df.groupby("part_id").substitute_part_id.apply(list).to_dict()


def available(part, use_substitutes, memo):
    if part not in memo:
        ok = stock.get(part, 0) > 0
        if not ok and kind[part] != "COMPONENT":
            ok = all(available(child, use_substitutes, memo) for child in bom_graph.successors(part))
        if not ok and use_substitutes:
            ok = any(stock.get(s, 0) > 0 for s in alternatives.get(part, []))
        memo[part] = ok
    return memo[part]


for use_subs in (False, True):
    memo = {}
    print("with substitutes" if use_subs else "without        ",
          sorted(r.product_id for r in products_df.itertuples() if available(r.part_id, use_subs, memo)))

In [ ]:
# PyRel: the same positive formulation, as rules that stay in the model.
Part.is_short_strict = model.Relationship(f"{Part} is short ignoring substitutes")
model.where(Part.kind == "COMPONENT", Part.on_hand == 0).define(Part.is_short_strict())

Part.has_available_substitute = model.Relationship(f"{Part} has a substitute in stock")
original, alt, ratio = Part.ref(), Part.ref(), Float.ref()
model.where(original.substitute(alt, ratio), alt.on_hand > 0).define(original.has_available_substitute())

Part.is_short = model.Relationship(f"{Part} is short")
model.where(Part.is_short_strict(), model.not_(Part.has_available_substitute())).define(Part.is_short())

# "blocked" propagates upwards through out-of-stock modules: positive recursion, no negation inside.
Part.is_blocked_strict = model.Relationship(f"{Part} cannot be built ignoring substitutes")
Part.is_blocked = model.Relationship(f"{Part} cannot be built")
parent, child, q = Part.ref(), Part.ref(), Integer.ref()
model.where(Part.is_short_strict()).define(Part.is_blocked_strict())
model.where(parent.on_hand == 0, parent.contains(child, q), child.is_blocked_strict()).define(parent.is_blocked_strict())
model.where(Part.is_short()).define(Part.is_blocked())
model.where(parent.on_hand == 0, parent.contains(child, q), child.is_blocked()).define(parent.is_blocked())

t1, t2 = Part.ref(), Part.ref()
without = sorted(model.where(Product.top_part(t1), model.not_(t1.is_blocked_strict())).select(
    Product.id.alias("p")).to_df()["p"])
with_subs = sorted(model.where(Product.top_part(t2), model.not_(t2.is_blocked())).select(
    Product.id.alias("p")).to_df()["p"])
print("without substitutes:", without)
print("with substitutes:   ", with_subs, "-> rescued:", sorted(set(with_subs) - set(without)))
assert without == EXPECTED["buildable_without_substitutes"]
assert with_subs == EXPECTED["buildable_with_substitutes"]

**Verdict:** the substitute rescues `PR-04`, the planted case. It also rescues `PR-09`, through one of the randomly generated substitute pairs. All three approaches agree. SQL and PyRel both needed the same positive reformulation; the gain from PyRel is that `is_blocked` is now a named fact in the model.

## 5. Which suppliers are structurally critical?

So far every question had a precise answer you could, with effort, write in SQL. *Structural* questions are different: which suppliers sit on the most paths between others (betweenness centrality), or which groups of suppliers form clusters (community detection)? **SQL has no graph algorithms.** Python has NetworkX, in memory. RelationalAI has a graph reasoner that runs on the model we already built.

We treat the supplier network as undirected, because a shared supplier links two firms either way. The planted cycle means `S-T2-11` and `S-T2-14` are linked twice; `aggregator="sum"` merges that into one edge.

In [ ]:
from relationalai.semantics.reasoners.graph import Graph

graph = Graph(model, directed=False, weighted=False, node_concept=Supplier, aggregator="sum")
g1, g2 = Supplier.ref(), Supplier.ref()
model.where(g1.depends_on(g2)).define(graph.Edge.new(src=g1, dst=g2))

# Algorithm results become properties of Supplier (the attribute name must match the algorithm).
Supplier.betweenness_centrality = graph.betweenness_centrality()
Supplier.community_assignments = graph.louvain()

critical = model.select(
    Supplier.id.alias("supplier"), Supplier.tier.alias("tier"), Supplier.region.alias("region"),
    Supplier.betweenness_centrality.alias("betweenness"), Supplier.community_assignments.alias("community"),
).to_df()
critical["community"] = critical["community"].astype(str)
display(critical.sort_values("betweenness", ascending=False).head(10))
print("communities found:", critical["community"].nunique(), "(Louvain is not deterministic; the count can vary)")

In [ ]:
# Python alternative and cross-check: NetworkX gives the same ranking (scores agree within ~1%).
g = supplier_graph.to_undirected()
g.add_nodes_from(table("SUPPLIERS").supplier_id)
nx_bc = pd.Series(nx.betweenness_centrality(g, normalized=False), name="networkx")
compare = critical.set_index("supplier")[["betweenness"]].join(nx_bc)
display(compare.sort_values("betweenness", ascending=False).head(10).round(1))
top5 = lambda col: list(compare.sort_values(col, ascending=False).index[:5])
assert top5("betweenness") == top5("networkx")

**Verdict:** this is outside SQL. NetworkX and RelationalAI rank the suppliers the same way. With NetworkX the graph is built in notebook memory from pulled DataFrames. With RelationalAI the scores become properties of `Supplier` in the model, ready for rules or optimisation.

Note that `S-T3-07` doesn't top the ranking. Betweenness measures how much a supplier *connects* others, not how much depends on it, and `S-T3-07` has only three links. Its risk only appeared in chapter 3, by following dependencies through to products. Different questions need different tools.

## 6. Which orders should get the scarce sensors? (case 2, continued)

With `S-T3-07` down, no new `P-141` sensors arrive, and the control board is out of stock. All that remains is the **2,960 sensors in stock**. Each control board needs 2. The 568 affected orders together need 29,286 sensors. Which orders do we fulfil to keep the most order value?

That's a decision with a constraint: an optimisation problem, and **outside SQL**. In Python you'd build the model by hand from DataFrames with a solver library (PuLP, OR-Tools). In PyRel we add decision variables to the same model and reuse its facts. Here it's a knapsack: one yes/no decision per order, a sensor budget, maximise value.

In [ ]:
from relationalai.semantics.reasoners.prescriptive import Problem

SENSOR, BOARD, SENSORS_PER_BOARD = "P-141", "P-040", 2.0   # the control board holds 2 x P-141

# Facts the solver needs, derived from the model: boards per unit of each product (direct or one level down) ...
Product.boards_per_unit = model.Property(f"{Product} needs {Integer:boards_per_unit} control boards per unit")
t, mid, brd, q1, q2 = Part.ref(), Part.ref(), Part.ref(), Integer.ref(), Integer.ref()
model.where(Product.top_part(t), t.contains(brd, q1), brd.id == BOARD).define(Product.boards_per_unit(q1))
model.where(Product.top_part(t), t.contains(mid, q1), mid.contains(brd, q2), brd.id == BOARD).define(
    Product.boards_per_unit(q1 * q2))
# ... and sensors needed per order.
Order.sensors_needed = model.Property(f"{Order} needs {Float:sensors_needed} temperature sensors")
model.where(Order.product(Product), Product.boards_per_unit(q1)).define(
    Order.sensors_needed(Order.quantity * q1 * SENSORS_PER_BOARD))

sensor_stock = float(model.where(Part.id == SENSOR).select(Part.on_hand.alias("s")).to_df()["s"].iloc[0])

# Decision: fill an order (1) or not (0). Only orders that need sensors get a variable.
Order.x_fill = model.Property(f"{Order} is filled {Float:x_fill}")
problem = Problem(model, Float)
problem.solve_for(Order.x_fill, where=[Order.sensors_needed], type="bin", name=["fill", Order.id])
problem.satisfy(model.require(aggs.sum(Order.x_fill * Order.sensors_needed) <= sensor_stock))
problem.maximize(aggs.sum(Order.x_fill * Order.value))
problem.solve("highs", time_limit_sec=60)   # the first solve starts a solver engine: allow a few minutes
print("status:", problem.solve_info().termination_status)

plan = model.where(Order.x_fill > 0.5, Order.product(Product)).select(
    Order.id.alias("order"), Product.id.alias("product"), Order.quantity.alias("qty"),
    Order.sensors_needed.alias("sensors"), Order.value.alias("value")).to_df()
print(f"fill {len(plan)} of 568 orders, using {plan.sensors.sum():.0f} of {sensor_stock:.0f} sensors, "
      f"keeping {plan.value.sum():,.2f} of {EXPECTED['affected_order_value']:,.2f} in order value")
display(plan.sort_values("value", ascending=False).head(10))

**Verdict:** when we tested this, the solver filled 48 of the 568 orders, used all 2,960 sensors, and kept 1,136,506.80 in order value (status `OPTIMAL`). The facts it needed, "which orders contain the board" and "how many sensors each needs", came from the same model as chapters 1–3.

**Possible extension:** let the chapter 4 rules restrict the candidates. Only fill orders whose product isn't blocked by *other* shortages, so every allocated sensor actually turns into a shipped product.

## 7. Report the results with SQL

Back to the right tool for the last step. Write the results to tables, and anyone can query them with SQL, a dashboard or Cortex.

In [ ]:
# Results from the model become ordinary tables.
# RelationalAI returns integer columns as Int128, which can't be uploaded: cast them first.
columns = {"order": "ORDER_ID", "product": "PRODUCT_ID", "value": "ORDER_VALUE",
           "qty": "QUANTITY", "sensors": "SENSORS_ALLOCATED"}
plan_out = plan.astype({"qty": "int64", "sensors": "float64", "value": "float64"}).rename(columns=columns)
risk_out = affected_rai.astype({"value": "float64"}).rename(columns=columns)
session.write_pandas(risk_out, "ORDERS_AT_RISK", auto_create_table=True, overwrite=True)
session.write_pandas(plan_out, "SENSOR_ALLOCATION", auto_create_table=True, overwrite=True)

sql("""
SELECT c.region,
       COUNT(*)                         AS orders_at_risk,
       SUM(r.order_value)               AS value_at_risk,
       COUNT(a.order_id)                AS orders_filled,
       COALESCE(SUM(a.order_value), 0)  AS value_kept
FROM ORDERS_AT_RISK r
JOIN ORDERS o ON o.order_id = r.order_id
JOIN CUSTOMERS c ON c.customer_id = o.customer_id
LEFT JOIN SENSOR_ALLOCATION a ON a.order_id = r.order_id
GROUP BY c.region ORDER BY value_at_risk DESC
""")

## Summary: which tool for which question

| Chapter | Question | SQL | Python (pandas/NetworkX) | RelationalAI |
|---|---|---|---|---|
| 0, 7 | Look around, report | **Best fit** | | |
| 2 | Contains at any depth | Recursive CTE, fine | Graph walk in memory | Recursive rule, reused later |
| 3 | Supplier failure through both networks | Works, but needs a hand-written cycle guard | Works; glue between two graphs | Rules on the same model; cycle needs no special handling |
| 4 | Buildable from stock | Works once reformulated positively | Recursive function | Same reformulation; result stays in the model |
| 5 | Structural criticality | **Not possible** | NetworkX, in memory | Graph reasoner; scores become properties |
| 6 | Allocate scarce stock | **Not possible** | Solver library, model built by hand | Prescriptive reasoner on the same model |

**What the model buys you:** one definition of the business — parts, suppliers, orders and how they connect — that rules, graph algorithms and optimisation all build on. The alternative is restating join paths in every query, or rebuilding DataFrames and graphs in every script.

**What it costs:**
- The modelling step in chapter 1.
- A separate engine, with a few minutes' start-up on the first query.
- One more tool to learn.

**About scale:** this dataset is deliberately small, so the Python alternative works fine here. With Python, the data is pulled into the notebook kernel and you manage sizing yourself. RelationalAI runs in its own Snowflake-managed engine, sized by configuration, and reads the tables via change data capture. Both are bounded by memory; the difference is where the work runs and who manages it.

**Next steps:**
- [RelationalAI templates](https://docs.relational.ai/build/templates)
- [Build with RelationalAI](https://docs.relational.ai/build)
- Change `FAILED` in chapter 3 to another supplier and rerun chapters 3 and 6.